## Project Introduction
As part of the data-engineering team at an e-commerce company specializing in **technology market insights**, your primary task is to design and implement an end-to-end **data pipeline** and **analysis workflow**. The company aims to analyze trends in the stock market for **prominent AI technology companies** to identify **market trends** and sell **actionable insights** to technology-focused businesses.

We chose to use **Yahoo Finance** as it provides reliable and extensive data on AI-related technology stocks, enabling:

1. Tracking stock performance over time.
2. Analyzing market trends and sector behavior.
3. Generating actionable insights for technology companies.

In [1]:
# Getting API access Yahoo Finance
import yfinance as yf
import numpy as np
import pandas as pd
import json
import redis
import pandas_gbq
from datetime import datetime
from google.cloud import bigquery
from io import StringIO

# Fetch list of ticker data from Yahoo Finance
tickers = ["MSFT", "GOOG", "META"]
tickersData = yf.download(tickers, start="2020-01-01", end="2025-05-30")



YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  3 of 3 completed


We chose **Redis**, as it is a **high-performance in-memory database** that is widely used for various data engineering and analytics workflows due to its **speed, versatility, and scalability**:
1. Caching: Store frequently accessed data to reduce computation or API overhead.

2. Real-Time Processing: Handle stock price data updates and analytics with minimal delay.

3. Temporary Storage: Act as a fast intermediate store for ETL workflows.

4. High Availability: Use Redis replication and clustering to ensure fault tolerance and scalability.

In [ ]:
# Connect to Redis
redisClient = redis.Redis(
    host='',
    port=13519,
    decode_responses=True,
    username="",
    password="",
)

We chose to save the data in **CSV** format, as it provides:

1. Simplicity for data extraction and transformation.

2. Compatibility with BigQuery and other tools in your workflow.

3. Portability and ease of debugging.

In [3]:
# Save as CSV
tickersDataCSV = tickersData.to_csv(index = True)

# Store data into Redis
redisClient.set(f"{tickers}DataCSV", tickersDataCSV)

# Fetch MSFT data from Redis 
# redisClient.get(f"{tickers}DataCSV")

True

In [4]:
# #Setup BigQuery Client
BQClient = bigquery.Client()

# #Transform data
tickersDataDF = pd.read_csv(StringIO(tickersDataCSV))
tickersDataDF.columns = [
    col.replace(".", "_") for col in tickersDataDF.columns  # Replace dots with underscores
]



# #Load DF into BigQuery
projectID = "stroff-1130"
datasetID = "NTUProj"
tableID = f"{projectID}.{datasetID}.tickers"


# Load DataFrame into BigQuery
try:
    pandas_gbq.to_gbq(
        tickersDataDF,
        destination_table=tableID,
        project_id=projectID,
        if_exists="replace"  # Options: 'fail', 'replace', 'append'
    )
    print(f"Data successfully loaded into {tableID}.")
except Exception as e:
    print(f"Error loading data into BigQuery: {e}")



100%|██████████| 1/1 [00:00<00:00, 4310.69it/s]

Data successfully loaded into stroff-1130.NTUProj.tickers.
